# Notebook 01 — Clean Connectome Build (Witvliet 2020)

## Goal
Produce a single, reproducible C. elegans adult hermaphrodite adjacency from
Witvliet et al. 2020. Separate chemical and gap-junction layers. No dense-edgelist.
No L/R collapse (kept as sensitivity analysis only).

## Preregistered success criteria (corrected 2026-04-17)

**Correction note:** Initial draft used White 1986 priors (~2.5k chem edges, 302
neurons). Witvliet 2020 is a denser EM reconstruction covering a partial neuron
set with higher per-edge synapse counts. Bounds below are calibrated to Witvliet
2020 adult reconstruction as the primary source (the file we are actually loading):

| Quantity | Witvliet 2020 adult | Preregistered bound |
|---|---|---|
| Chemical edges (distinct (pre,post) pairs) | 2202 | `2000 ≤ edges ≤ 2500` |
| Chemical total synapse contacts | 7467 | `6500 ≤ total ≤ 8500` |
| Electrical edges | 291 | `260 ≤ edges ≤ 330` |
| Electrical total synapses (pre-mirror) | 400 | `350 ≤ total ≤ 500` |
| Unique neurons (L/R distinct) | 222 | `210 ≤ N ≤ 235` |

Plus structural sanity:

- **No dense-edgelist artifact**: no neuron has out-degree equal to N−1 in the
  chemical adjacency (that was the fingerprint of the old `connectome_edges_canonical.csv`).
- **Chem density** < 0.10 (real connectome is ~0.045).
- **Max in-degree** ≥ 20 (a populated matrix has highly-targeted hub neurons).

## Halting rule
If any criterion fails, `assert all_pass` halts the notebook. **Do not proceed to
Notebook 02 until all criteria pass.** If bounds need revising, the revision
must be documented as a further correction note (not a silent edit).

## Outputs (only written if all criteria pass)
- `data_derived/connectome_adult.npz` — canonical numpy arrays
- `data_derived/connectome_adult_edges.csv` — tidy edges
- `data_derived/connectome_adult_summary.csv` — summary statistics
- `data_derived/developmental/` — all six stages


In [1]:
# Setup — import the shared loader from lib/
import sys
from pathlib import Path
_HERE = Path.cwd()
if (_HERE / "lib").is_dir():
    sys.path.insert(0, str(_HERE))
elif (_HERE.parent / "lib").is_dir():
    sys.path.insert(0, str(_HERE.parent))
else:
    raise RuntimeError("Cannot locate lib/ — run this notebook from within New Notebooks/")

from lib.connectome import load_witvliet, load_all_stages, save_connectome
from lib.paths import DERIVED, WITVLIET_FILES

import numpy as np
import pandas as pd

print("lib imports OK")
print(f"Derived output dir: {DERIVED}")
print(f"Witvliet stages available: {list(WITVLIET_FILES.keys())}")


lib imports OK
Derived output dir: /mnt/ssd4tb/Desktop/C-Elegans/New Notebooks/data_derived
Witvliet stages available: ['L1_1', 'L1_2', 'L1_3', 'L1_4', 'L2', 'L3', 'adult']


## Step 1 — Load the adult connectome

In [2]:
adult = load_witvliet("adult")

print(f"Stage: {adult.stage}")
print(f"Unique neurons (L/R distinct): {adult.n}")
print(f"Source rows after self-loop removal: {len(adult.edges_raw)}")
print(f"Self-loops dropped: {adult.edges_raw.attrs['self_loops_dropped']}")
print(f"Chem adjacency: shape {adult.chem_adj.shape}, dtype {adult.chem_adj.dtype}")
print(f"Gap  adjacency: shape {adult.gap_adj.shape}, dtype {adult.gap_adj.dtype}")


Stage: adult
Unique neurons (L/R distinct): 222
Source rows after self-loop removal: 2476
Self-loops dropped: 17
Chem adjacency: shape (222, 222), dtype int32
Gap  adjacency: shape (222, 222), dtype int32


## Step 2 — Evaluate against preregistered criteria

In [3]:
chem_total = int(adult.chem_adj.sum())
chem_edges = int((adult.chem_adj > 0).sum())
gap_edges_unique = int((adult.gap_adj > 0).sum()) // 2          # loader mirrors; // 2 gives pre-mirror count
gap_total_unique = int(adult.gap_adj.sum()) // 2                # same: sum is doubled by mirror
N = adult.n
chem_density = chem_edges / (N * (N - 1))
max_out_deg = int((adult.chem_adj > 0).sum(axis=1).max())
max_in_deg  = int((adult.chem_adj > 0).sum(axis=0).max())

print(f"chem_edges (distinct directed):     {chem_edges}")
print(f"chem_total (sum of synapse counts): {chem_total}")
print(f"gap_edges_unique:                   {gap_edges_unique}")
print(f"gap_total_unique (pre-mirror sum):  {gap_total_unique}")
print(f"N (unique neurons):                 {N}")
print(f"chem_density:                       {chem_density:.4f}")
print(f"max_out_deg:                        {max_out_deg}  (= N-1 would indicate dense-edgelist)")
print(f"max_in_deg:                         {max_in_deg}")


chem_edges (distinct directed):     2191
chem_total (sum of synapse counts): 7455
gap_edges_unique:                   285
gap_total_unique (pre-mirror sum):  394
N (unique neurons):                 222
chem_density:                       0.0447
max_out_deg:                        36  (= N-1 would indicate dense-edgelist)
max_in_deg:                         40


In [4]:
checks = [
    ("1 chem_edges in [2000, 2500]",            2000 <= chem_edges <= 2500,             f"chem_edges={chem_edges}"),
    ("2 chem_total in [6500, 8500]",            6500 <= chem_total <= 8500,             f"chem_total={chem_total}"),
    ("3 gap_edges_unique in [260, 330]",        260  <= gap_edges_unique <= 330,        f"gap_edges_unique={gap_edges_unique}"),
    ("4 gap_total_unique in [350, 500]",        350  <= gap_total_unique <= 500,        f"gap_total_unique={gap_total_unique}"),
    ("5 N in [210, 235]",                       210  <= N <= 235,                       f"N={N}"),
    ("6 no degree=N-1 artifact",                max_out_deg < N - 1,                    f"max_out_deg={max_out_deg}, N-1={N-1}"),
    ("7 chem density < 0.10",                   chem_density < 0.10,                    f"chem_density={chem_density:.4f}"),
    ("8 max_in_deg >= 20",                      max_in_deg >= 20,                       f"max_in_deg={max_in_deg}"),
]

print("PREREGISTERED CRITERIA")
print("=" * 60)
all_pass = True
for label, passed, detail in checks:
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {label:42s}  {detail}")
    if not passed:
        all_pass = False

print("=" * 60)
print(f"ALL CRITERIA PASS: {all_pass}")
assert all_pass, "Halting: preregistered criteria not all met. Fix the loader before proceeding."


PREREGISTERED CRITERIA
  [PASS] 1 chem_edges in [2000, 2500]                chem_edges=2191
  [PASS] 2 chem_total in [6500, 8500]                chem_total=7455
  [PASS] 3 gap_edges_unique in [260, 330]            gap_edges_unique=285
  [PASS] 4 gap_total_unique in [350, 500]            gap_total_unique=394
  [PASS] 5 N in [210, 235]                           N=222
  [PASS] 6 no degree=N-1 artifact                    max_out_deg=36, N-1=221
  [PASS] 7 chem density < 0.10                       chem_density=0.0447
  [PASS] 8 max_in_deg >= 20                          max_in_deg=40
ALL CRITERIA PASS: True


## Step 3 — Save adult artifacts

In [5]:
paths = save_connectome(adult, DERIVED)
for k, p in paths.items():
    print(f"  {k}: {p}  ({p.stat().st_size/1024:.1f} KB)")


  npz: /mnt/ssd4tb/Desktop/C-Elegans/New Notebooks/data_derived/connectome_adult.npz  (7.8 KB)
  edges_csv: /mnt/ssd4tb/Desktop/C-Elegans/New Notebooks/data_derived/connectome_adult_edges.csv  (53.3 KB)
  summary_csv: /mnt/ssd4tb/Desktop/C-Elegans/New Notebooks/data_derived/connectome_adult_summary.csv  (0.2 KB)


## Step 4 — Load and validate the developmental series

Witvliet's four L1 replicates, L2, L3, adult. Reasonable expectation: connectome
grows over development; adult should have more chemical synapses than any L1 replicate.

In [6]:
stages = load_all_stages()
rows = [c.summary() for c in stages.values()]
dev_summary = pd.DataFrame(rows).set_index("stage")
dev_summary


,n_neurons,chem_total_synapses,gap_total_synapses,chem_directed_edges,gap_unique_undirected_edges,chem_mean_syn_per_edge,density_chem
stage,,,,,,,
L1_1,187,1296,204,775,81,1.672258,0.022282
L1_2,194,1895,262,986,122,1.921907,0.026334
L1_3,198,2128,214,1012,92,2.102767,0.025945
L1_4,204,2777,480,1136,206,2.444542,0.027432
L2,211,4116,826,1515,286,2.716832,0.034191
L3,216,4456,530,1525,212,2.921967,0.032838
adult,222,7455,788,2191,285,3.402556,0.044658


In [7]:
# Developmental-growth sanity: adult > all L1 replicates (on chemical synapses)
adult_chem = dev_summary.loc["adult", "chem_total_synapses"]
l1_chem = dev_summary.loc[["L1_1", "L1_2", "L1_3", "L1_4"], "chem_total_synapses"].values
print(f"adult chem: {adult_chem}")
print(f"L1 chem:    {l1_chem.tolist()}  (mean {l1_chem.mean():.0f})")
assert adult_chem > l1_chem.max(), (
    "Unexpected: an L1 replicate has more chemical synapses than adult. "
    "Violates developmental-growth expectation."
)
print("Monotonic-growth check: PASS")


adult chem: 7455
L1 chem:    [1296, 1895, 2128, 2777]  (mean 2024)
Monotonic-growth check: PASS


In [8]:
# Save developmental series
dev_dir = DERIVED / "developmental"
dev_dir.mkdir(exist_ok=True)
for stage, conn in stages.items():
    save_connectome(conn, dev_dir)
dev_summary.to_csv(DERIVED / "developmental_summary.csv")
print(f"Saved {len(stages)} stages to {dev_dir}")


Saved 7 stages to /mnt/ssd4tb/Desktop/C-Elegans/New Notebooks/data_derived/developmental


## Step 5 — Sensitivity analysis: L/R collapse

Per (D1) in the loader, the default is L/R-distinct. Running with `collapse_lr=True`
is a sensitivity analysis only. We document the difference so downstream notebooks
can choose deliberately.

In [9]:
adult_collapsed = load_witvliet("adult", collapse_lr=True)
comparison = pd.DataFrame([adult.summary(), adult_collapsed.summary()]).T
comparison.columns = ["L/R distinct (default)", "L/R collapsed (sensitivity)"]
comparison


,L/R distinct (default),L/R collapsed (sensitivity)
stage,adult,adult
n_neurons,222,137
chem_total_synapses,7455,7347
gap_total_synapses,788,708
chem_directed_edges,2191,1366
gap_unique_undirected_edges,285,175
chem_mean_syn_per_edge,3.402556,5.378477
density_chem,0.044658,0.073315


## Step 6 — Final statement

If every preregistered check passed, `data_derived/connectome_adult.npz` is the
**single source of truth** for the adult connectome for every downstream notebook.
Any downstream notebook that builds its own adjacency from the raw files must
justify the deviation in a preregistered markdown cell.